# Session 3: Available View Types

Vitessce ships with a large number of interactive visualization and control views. In this notebook, we look at some types of views that you may not be familiar with.


The views available in Vitessce can be categorized into:
- Spatial view (covered in notebook 01)
- Scatterplot view for dimensionality reductions / embeddings
- Heatmap for expression data
- Control views
  - Cell set manager
  - Feature list
  - Spatial layer controller
- Statistical plots
  - Cell set sizes bar plot
  - Expression distribution per cell set violin plots
  - Expression distribution histogram
  - Cell set and sample set sizes treemap
  - Volcano plot
  - Cell type composition bar plot
  - Gene set enrichment bar plot
  - Gating scatterplot


We first show how to access these view types through the lower-level `VitessceConfig` API. Then, we use the EasyVitessce API for more concise configuration. The list of view types available via EasyVitessce can be found at the [documentation](https://vitessce.github.io/easy_vitessce/easy_vitessce.html).

In [ ]:
!pip install "vitessce[all]==3.9.2" "scanpy" "easy-vitessce==0.0.12" "spatialdata==0.7.3" "spatialdata-plot==0.4.0"

In [ ]:
import scanpy as sc
import os
from os.path import join, isdir

from vitessce import (
    VitessceConfig,
    ViewType as vt,
    AnnDataWrapper,
    SpatialDataWrapper,
    vconcat, hconcat,
)
from vitessce.data_utils import VAR_CHUNK_SIZE

## Load the data

In [ ]:
adata = sc.datasets.pbmc68k_reduced()

# Run t-SNE to be able to demonstrate sc.pl.tsne.
sc.tl.tsne(adata)

zarr_path = join("data", "pbmc68k.zarr")
if not isdir(zarr_path):
    os.makedirs("data", exist_ok=True)
    adata.write_zarr(zarr_path, chunks=[adata.shape[0], VAR_CHUNK_SIZE])

adata

## Embedding scatterplot

In [ ]:
vc_scatterplot = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc_scatterplot.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_set_paths=["obs/bulk_labels", "obs/louvain"],
        obs_set_names=["Cell Type", "Leiden Cluster"],
        obs_embedding_paths=["obsm/X_umap", "obsm/X_pca"],
        obs_embedding_names=["UMAP", "PCA"],
        obs_feature_matrix_path="X",
    )
)

# --- Define views ---
umap      = vc_scatterplot.add_view(vt.SCATTERPLOT, dataset=dataset)
obs_sets  = vc_scatterplot.add_view(vt.OBS_SETS, dataset=dataset)
genes     = vc_scatterplot.add_view(vt.FEATURE_LIST, dataset=dataset)

vc_scatterplot.link_views_by_dict([umap], {
    "embeddingType": "UMAP",
    "embeddingObsSetLabelsVisible": False,
})

# --- Define the layout ---
vc_scatterplot.layout(umap | (obs_sets / genes))
vc_scatterplot.widget()

## Heatmap

In [ ]:
vc_heatmap = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc_heatmap.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_set_paths=["obs/bulk_labels", "obs/louvain"],
        obs_set_names=["Cell Type", "Leiden Cluster"],
        obs_feature_matrix_path="X",
    )
)

# --- Define views ---
heatmap   = vc_heatmap.add_view(vt.HEATMAP, dataset=dataset).set_props(transpose=True)
obs_sets  = vc_heatmap.add_view(vt.OBS_SETS, dataset=dataset)

vc_heatmap.link_views_by_dict([heatmap], { "featureValueColormapRange": [0.0, 0.25] })

# --- Define the layout ---
vc_heatmap.layout(heatmap | obs_sets)
vc_heatmap.widget()

## Cell set sizes bar plot

In [ ]:
vc_barplot = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc_barplot.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_set_paths=["obs/bulk_labels", "obs/louvain"],
        obs_set_names=["Cell Type", "Leiden Cluster"],
    )
)

# --- Define views ---
obs_set_sizes   = vc_barplot.add_view(vt.OBS_SET_SIZES, dataset=dataset)
obs_sets  = vc_barplot.add_view(vt.OBS_SETS, dataset=dataset)

# --- Define the layout ---
vc_barplot.layout(obs_set_sizes | obs_sets)
vc_barplot.widget()

## Histogram

In [ ]:
vc_histogram = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc_histogram.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_feature_matrix_path="X",
    )
)

# --- Define views ---
histogram   = vc_histogram.add_view(vt.FEATURE_VALUE_HISTOGRAM, dataset=dataset)
genes  = vc_histogram.add_view(vt.FEATURE_LIST, dataset=dataset)

# --- Define the layout ---
vc_histogram.layout(histogram | genes)
vc_histogram.widget()

## Violin plots

In [ ]:
vc_violin = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc_violin.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_set_paths=["obs/bulk_labels", "obs/louvain"],
        obs_set_names=["Cell Type", "Leiden Cluster"],
        obs_feature_matrix_path="X",
    )
)

# --- Define views ---
violins   = vc_violin.add_view(vt.OBS_SET_FEATURE_VALUE_DISTRIBUTION, dataset=dataset)
obs_sets   = vc_violin.add_view(vt.OBS_SETS, dataset=dataset)
genes  = vc_violin.add_view(vt.FEATURE_LIST, dataset=dataset)

vc_violin.link_views_by_dict([violins, obs_sets, genes], { "featureSelection": ["CLIC1"] })

# --- Define the layout ---
vc_violin.layout(violins | (obs_sets/genes))
vc_violin.widget()

## Dot plot

In [ ]:
vc_dot = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc_dot.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_set_paths=["obs/bulk_labels", "obs/louvain"],
        obs_set_names=["Cell Type", "Leiden Cluster"],
        obs_feature_matrix_path="X",
    )
)

# --- Define views ---
dot_plot   = vc_dot.add_view("dotPlot", dataset=dataset)
obs_sets   = vc_dot.add_view(vt.OBS_SETS, dataset=dataset)
genes  = vc_dot.add_view(vt.FEATURE_LIST, dataset=dataset).set_props(enableMultiSelect=True)

vc_dot.link_views_by_dict([dot_plot, obs_sets, genes], { "featureSelection": ["ABI3", "ACAA1", "ACADS"] })

# --- Define the layout ---
vc_dot.layout(dot_plot | (obs_sets/genes))
vc_dot.widget()

# The EasyVitessce approach

With EasyVitessce, many of the examples above can be expressed with a single line of code.

In [ ]:
import easy_vitessce as ev

In [ ]:
# Begin Colab-specific lines. Not required when running locally.
# Reference: https://vitessce.github.io/easy_vitessce/customization.html
ev.register_data_path(adata, zarr_path)
ev.config.set({ 'data.wrapper_param_suffix': '_store' })
# End Colab-specific lines.

## Embedding scatterplot

Colored by cell type:

In [ ]:
sc.pl.embedding(adata, basis="umap", color="bulk_labels")

Color by gene expression value:

In [ ]:
sc.pl.embedding(adata, basis="umap", color="ABI3")

Color by both expression value and cell type label:

In [ ]:
sc.pl.embedding(adata, basis="umap", color=["ABI3", "bulk_labels"])

## Heatmap

In [ ]:
adata.var.head()

In [ ]:
sc.pl.heatmap(adata, groupby="bulk_labels", var_names=adata.var.loc[adata.var["highly_variable"]].index.tolist(), vmax=0.1)

## Violin plot

With one gene selected:

In [ ]:
sc.pl.violin(adata, groupby="bulk_labels", keys=["CD74"])

With multiple genes selected:

In [ ]:
sc.pl.violin(adata, groupby="bulk_labels", keys=["CD74", "CAP1"])

## Dot plot

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=["C1QA", "PSAP", "CD79A", "CD79B", "CST3", "LYZ", "ANXA1", "S100A4"],
    groupby="bulk_labels",
)